# Part 1: Knowledge bases and Microsoft Retriever 1

Cocoarynth Chocolate Works is a small bean-to-bar chocolate maker. It wants to print an organic seal on its wrappers, like this one:

<img src="cocoarynth_organic_bar.jpg" alt="A Cocoarynth chocolate bar wrapper with a green organic seal in the top right corner" width="400">

When is a bar allowed to carry that seal? The answer is somewhere in Cocoarynth's records, which are scattered across farm logs, factory test results, shipping paperwork, retailer emails, and company policies. In this notebook, you'll build a knowledge base over those records and ask it.

**🎯 Mission**
- Connect three search indexes as **indexed knowledge sources** and combine them in a **knowledge base**.
- Ask about the organic seal, then check the answer's citations against the original document.
- Compare standard agentic retrieval with **Microsoft Retriever 1 (MR1)**, a faster retriever that searches every source instead of choosing.

## The building blocks

In Foundry IQ, a **knowledge base** is a top-level resource that connects a chat completion model to one or more knowledge sources. It defines which data sources to query, which model to use for reasoning, and how query execution should be optimized.

An **indexed knowledge source** connects a knowledge base to a pre-built search index, giving you full control over its schema, chunking, and embeddings.

## Step 1: Load your settings

Run the cell below to load your settings from the `.env` file.

**ℹ️ Note**
- The first time you run the cell below, you'll be prompted to select a kernel. Select **Python Environments** and choose the **.venv** environment.
- You may also be prompted with "Do you want to allow public and private networks to access this app?" Select **Allow**.

**⚠️ Troubleshooting:**
If code cells get stuck and keep spinning, try these remediation steps:
1. Select **Restart** from the notebook toolbar at the top. If that doesn't work:
2. Select **Reload window** from the VS Code command palette. If that doesn't work:
3. Close VS Code completely and restart it.

In [ ]:
import os

from azure.core.credentials import AzureKeyCredential
from dotenv import load_dotenv

load_dotenv(override=True)

AZURE_SEARCH_SERVICE_ENDPOINT = os.environ["AZURE_SEARCH_SERVICE_ENDPOINT"]
AZURE_SEARCH_ADMIN_KEY = os.environ["AZURE_SEARCH_ADMIN_KEY"]
AZURE_OPENAI_ENDPOINT = os.environ["AZURE_OPENAI_ENDPOINT"]
AZURE_OPENAI_KEY = os.environ["AZURE_OPENAI_KEY"]
AZURE_OPENAI_CHATGPT_DEPLOYMENT = os.environ["AZURE_OPENAI_CHATGPT_DEPLOYMENT"]
AZURE_OPENAI_CHATGPT_MODEL_NAME = os.environ["AZURE_OPENAI_CHATGPT_MODEL_NAME"]

API_VERSION = "2026-11-01-preview"
search_credential = AzureKeyCredential(AZURE_SEARCH_ADMIN_KEY)
print("Environment variables loaded")

## Step 2: Check the indexes

The Foundry IQ agent plugin split the Cocoarynth PDFs into **chunks** ahead of time: pieces of text such as a section, a table, or a short page. It also computed an embedding for each chunk. Setup loaded them into one search index per document group:

* `cocoarynth-production`: the path from bean to bar, from farm, supplier, and harvest records to factory test results and batch records
* `cocoarynth-sales`: shipping paperwork, retailer agreements, and customer emails
* `cocoarynth-policies`: company policies

Setup also created three more indexes, `cocoarynth-chunks`, `cocoarynth-graph`, and `cocoarynth-topics`, which you'll use in the next notebooks.

Each chunk is stored as one search document, with the chunk's text in a field that's also named `chunk`. Run the cell below to count the chunks in each document index.

In [ ]:
from azure.search.documents import SearchClient

DOMAINS = ["production", "sales", "policies"]
INDEX_NAMES = {domain: f"cocoarynth-{domain}" for domain in DOMAINS}

for index_name in INDEX_NAMES.values():
    with SearchClient(
        endpoint=AZURE_SEARCH_SERVICE_ENDPOINT, index_name=index_name,
        credential=search_credential, api_version=API_VERSION,
    ) as search_client:
        print(f"{index_name}: {search_client.get_document_count()} PDF chunks")

## Step 3: Create knowledge sources and a knowledge base

The cell below creates three indexed knowledge sources, one per index. Each source searches the `chunk` field and returns citation fields such as `title`, `blob_path`, and page numbers.

It then creates a knowledge base named `cocoarynth-standard-knowledge-base`. A knowledge base is the orchestration layer that combines:

1. Your data sources (the three knowledge sources)
2. Models for planning queries and generating answers (the managed MR1 retriever and your Azure OpenAI chat model)
3. Configuration for how to process queries and format responses

![Diagram: cocoarynth-standard-knowledge-base uses MR1 and gpt-6-luna to plan queries and write a cited answer. It chooses which of three knowledge sources to search; each source searches its own index: cocoarynth-production, cocoarynth-sales, and cocoarynth-policies.](../../img/diagrams/notebook-01-standard.svg)

This knowledge base uses `ANSWER_SYNTHESIS` output mode, so the chat model writes a complete answer from the retrieved chunks. If you run the cell again, it updates the objects you already created.

In [ ]:
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    AzureOpenAIVectorizerParameters,
    KnowledgeBase,
    KnowledgeBaseAzureOpenAIModel,
    KnowledgeBaseMicrosoftRetriever1Model,
    KnowledgeSourceReference,
    SearchIndexFieldReference,
    SearchIndexKnowledgeSource,
    SearchIndexKnowledgeSourceParameters,
)
from azure.search.documents.knowledgebases.models import (
    KnowledgeRetrievalLowReasoningEffort,
    KnowledgeRetrievalOutputMode,
)

index_client = SearchIndexClient(
    AZURE_SEARCH_SERVICE_ENDPOINT, search_credential, api_version=API_VERSION
)

KNOWLEDGE_SOURCE_NAMES = {domain: f"cocoarynth-{domain}-knowledge-source" for domain in DOMAINS}
for domain, knowledge_source_name in KNOWLEDGE_SOURCE_NAMES.items():
    knowledge_source = SearchIndexKnowledgeSource(
        name=knowledge_source_name,
        description=f"Cocoarynth {domain} document chunks",
        search_index_parameters=SearchIndexKnowledgeSourceParameters(
            search_index_name=INDEX_NAMES[domain],
            semantic_configuration_name="semantic-configuration",
            search_fields=[SearchIndexFieldReference(name="chunk")],
            source_data_fields=[
                SearchIndexFieldReference(name=field)
                for field in ("chunk_id", "title", "chunk", "blob_path", "page_number_from", "page_number_to")
            ],
        ),
    )
    index_client.create_or_update_knowledge_source(knowledge_source)
    print(f"Knowledge source '{knowledge_source_name}' created or updated.")

STANDARD_KB_NAME = "cocoarynth-standard-knowledge-base"
standard_knowledge_base = KnowledgeBase(
    name=STANDARD_KB_NAME,
    description="Standard agentic retrieval over the three Cocoarynth document indexes",
    knowledge_sources=[KnowledgeSourceReference(name=name) for name in KNOWLEDGE_SOURCE_NAMES.values()],
    models=[
        KnowledgeBaseMicrosoftRetriever1Model(),
        KnowledgeBaseAzureOpenAIModel(
            azure_open_ai_parameters=AzureOpenAIVectorizerParameters(
                resource_url=AZURE_OPENAI_ENDPOINT,
                deployment_name=AZURE_OPENAI_CHATGPT_DEPLOYMENT,
                model_name=AZURE_OPENAI_CHATGPT_MODEL_NAME,
                api_key=AZURE_OPENAI_KEY,
            )
        )
    ],
    retrieval_reasoning_effort=KnowledgeRetrievalLowReasoningEffort(),
    output_mode=KnowledgeRetrievalOutputMode.ANSWER_SYNTHESIS,
    answer_instructions="Answer from the retrieved records, cite reference IDs, and say when evidence is missing.",
)
index_client.create_or_update_knowledge_base(standard_knowledge_base)
print(f"Knowledge base '{STANDARD_KB_NAME}' created or updated.")

## Step 4: Ask a question

Ask the knowledge base when a bar is allowed to carry the organic seal. It plans one or more searches, retrieves chunks from the relevant sources, and writes an answer with citations like `[ref_id:0]`.

In [ ]:
from IPython.display import Markdown, display

from azure.search.documents.knowledgebases import KnowledgeBaseRetrievalClient
from azure.search.documents.knowledgebases.models import (
    KnowledgeBaseMessage,
    KnowledgeBaseMessageTextContent,
    KnowledgeBaseRetrievalRequest,
    SearchIndexKnowledgeSourceParams,
)

QUESTION = "When may a Cocoarynth bar carry the organic seal, and what if the supplier certificate is pending renewal?"

standard_client = KnowledgeBaseRetrievalClient(
    endpoint=AZURE_SEARCH_SERVICE_ENDPOINT,
    knowledge_base_name=STANDARD_KB_NAME,
    credential=search_credential,
    api_version=API_VERSION,
)
request = KnowledgeBaseRetrievalRequest(
    messages=[KnowledgeBaseMessage(role="user", content=[KnowledgeBaseMessageTextContent(text=QUESTION)])],
    retrieval_reasoning_effort=KnowledgeRetrievalLowReasoningEffort(),
    output_mode=KnowledgeRetrievalOutputMode.ANSWER_SYNTHESIS,
    include_activity=True,
    knowledge_source_params=[
        SearchIndexKnowledgeSourceParams(
            knowledge_source_name=name, include_references=True, include_reference_source_data=True
        )
        for name in KNOWLEDGE_SOURCE_NAMES.values()
    ],
)

result = standard_client.retrieve(request)
display(Markdown(result.response[0].content[0].text))

### Review the activity log

The activity log shows the steps the knowledge base took. Run the cell below to see them as a table with these columns:

| Field | Meaning |
| --- | --- |
| **id** | The activity ID, which connects each reference to the step that produced it. |
| **type** | The kind of step. You'll see `modelQueryPlanning` first, then a `searchIndex` step for each query sent to a knowledge source, `modelAnswerSynthesis` for writing the answer, and `agenticReasoning` for semantic reranking. |
| **knowledgeSourceName** | For search steps, which knowledge source was queried. For example, `cocoarynth-policies-knowledge-source` searches the `cocoarynth-policies` index. |
| **query** | For search steps, the query that the planning step wrote for that knowledge source. Each search can use different wording. |
| **count** | For search steps, how many results the search returned. |
| **elapsedMs** | How long the step took, in milliseconds. |

In [ ]:
from datetime import datetime


def elapsed_ms(step):
    """Return a step's duration. searchIndex steps currently report elapsedMs as 0, so use their timestamps instead."""
    if step.get("elapsedMs") or not (step.get("startedAt") and step.get("completedAt")):
        return step.get("elapsedMs", "")
    started, completed = (datetime.fromisoformat(step[key]) for key in ("startedAt", "completedAt"))
    return round((completed - started).total_seconds() * 1000)


rows = ["| id | type | knowledgeSourceName | query | count | elapsedMs |", "| --- | --- | --- | --- | --- | --- |"]
for activity in result.activity:
    step = activity.as_dict()
    rows.append(f"| {step['id']} | {step['type']} | {step.get('knowledgeSourceName', '')} | {step.get('searchIndexArguments', {}).get('search', '')} | {step.get('count', '')} | {elapsed_ms(step)} |")
display(Markdown("\n".join(rows)))

### Review the references

The references show which chunks were retrieved. Run the cell below to see them as a table with these columns:

| Field | Meaning |
| --- | --- |
| **id** | The number that appears in the answer's citation markers, like `[ref_id:0]`. |
| **activitySource** | The **id** of the search step in the activity log that returned it. |
| **rerankerScore** | The semantic reranker's score, from 0 to 4, where 4 is the most relevant. |
| **title** | The document the chunk came from. |
| **chunk** | The start of the retrieved chunk's text, exactly as it's stored in the index's `chunk` field, including any HTML from document extraction. |

In [ ]:
import textwrap

rows = ["| id | activitySource | rerankerScore | title | chunk |", "| --- | --- | --- | --- | --- |"]
for reference in result.references:
    source_data = reference.source_data or {}
    chunk = textwrap.shorten(source_data.get("chunk", ""), 400).replace("|", "\\|")
    rows.append(f"| {reference.id} | {reference.activity_source} | {reference.reranker_score:.2f} | {source_data.get('title')} | `{chunk}` |")
display(Markdown("\n".join(rows)))

### Check the answer

Open [the label and marketing claims policy](../../data/cocoarynth/documents/cocoarynth-policies/label-claims-policy.pdf) and skim section 4, "Organic claims." Does the answer match what the policy says about a certificate that's pending renewal? Which reference did that part of the answer come from?

## Step 5: Try Microsoft Retriever 1

Standard agentic retrieval uses a chat completion model to plan queries and choose which sources to search. **Microsoft Retriever 1 (MR1)** is built for speed: it plans queries without a chat completion model, and it doesn't choose sources. It searches every source in its knowledge base.

The cell below creates a second knowledge base, `cocoarynth-mr1-knowledge-base`, over the same three sources. It uses only MR1 and returns the matching chunks rather than writing an answer.

![Diagram: cocoarynth-mr1-knowledge-base uses only MR1 and returns chunks. It searches all three knowledge sources, each over its own index: cocoarynth-production, cocoarynth-sales, and cocoarynth-policies.](../../img/diagrams/notebook-01-mr1.svg)

In [ ]:
from azure.search.documents.knowledgebases.models import KnowledgeRetrievalMinimalReasoningEffort

MR1_KB_NAME = "cocoarynth-mr1-knowledge-base"

mr1_knowledge_base = KnowledgeBase(
    name=MR1_KB_NAME,
    description="Microsoft Retriever 1 evidence retrieval over the three Cocoarynth document indexes",
    knowledge_sources=[KnowledgeSourceReference(name=name) for name in KNOWLEDGE_SOURCE_NAMES.values()],
    models=[KnowledgeBaseMicrosoftRetriever1Model()],
    retrieval_reasoning_effort=KnowledgeRetrievalMinimalReasoningEffort(),
    output_mode=KnowledgeRetrievalOutputMode.EXTRACTIVE_DATA,
)
index_client.create_or_update_knowledge_base(mr1_knowledge_base)
print(f"Knowledge base '{MR1_KB_NAME}' created or updated.")

## Step 6: Run the same question through MR1

To make a fair comparison, both knowledge bases return chunks, not a written answer. The standard knowledge base gets the question as a chat message with low reasoning effort, and it chooses which of the three sources to search. MR1 gets it as a search intent with minimal reasoning effort, and it searches all three.

Run the cell and compare the activity log with the one from Step 4: you'll see a `searchIndex` step for every source. Then compare the returned chunks.

In [ ]:
from azure.search.documents.knowledgebases.models import KnowledgeRetrievalSemanticIntent, KnowledgeSourceParams

all_sources: list[KnowledgeSourceParams] = [
    SearchIndexKnowledgeSourceParams(
        knowledge_source_name=name, include_references=True, include_reference_source_data=True
    )
    for name in KNOWLEDGE_SOURCE_NAMES.values()
]


def standard_request():
    return KnowledgeBaseRetrievalRequest(
        messages=[KnowledgeBaseMessage(role="user", content=[KnowledgeBaseMessageTextContent(text=QUESTION)])],
        retrieval_reasoning_effort=KnowledgeRetrievalLowReasoningEffort(),
        output_mode=KnowledgeRetrievalOutputMode.EXTRACTIVE_DATA,
        include_activity=True,
        knowledge_source_params=all_sources,
        max_output_documents=8,
    )


def mr1_request():
    return KnowledgeBaseRetrievalRequest(
        intents=[KnowledgeRetrievalSemanticIntent(search=QUESTION)],
        retrieval_reasoning_effort=KnowledgeRetrievalMinimalReasoningEffort(),
        output_mode=KnowledgeRetrievalOutputMode.EXTRACTIVE_DATA,
        include_activity=True,
        knowledge_source_params=all_sources,
        max_output_documents=8,
    )


mr1_client = KnowledgeBaseRetrievalClient(
    endpoint=AZURE_SEARCH_SERVICE_ENDPOINT,
    knowledge_base_name=MR1_KB_NAME,
    credential=search_credential,
    api_version=API_VERSION,
)
mr1_result = mr1_client.retrieve(mr1_request())

rows = ["| id | type | knowledgeSourceName | query | count | elapsedMs |", "| --- | --- | --- | --- | --- | --- |"]
for activity in mr1_result.activity:
    step = activity.as_dict()
    rows.append(f"| {step['id']} | {step['type']} | {step.get('knowledgeSourceName', '')} | {step.get('searchIndexArguments', {}).get('search', '')} | {step.get('count', '')} | {elapsed_ms(step)} |")
display(Markdown("\n".join(rows)))

rows = ["| id | activitySource | rerankerScore | title | chunk |", "| --- | --- | --- | --- | --- |"]
for reference in mr1_result.references:
    source_data = reference.source_data or {}
    chunk = textwrap.shorten(source_data.get("chunk", ""), 200).replace("|", "\\|")
    rows.append(f"| {reference.id} | {reference.activity_source} | {reference.reranker_score:.2f} | {source_data.get('title')} | `{chunk}` |")
display(Markdown("\n".join(rows)))

## Step 7: Compare response times

The cell below sends the question to each knowledge base a few times and reports the median response time. It starts with one warm-up request per knowledge base that isn't counted.

These times include network latency and come from a small sample, so treat them as a rough demonstration rather than a benchmark. A faster response isn't necessarily a better one: compare the returned chunks too.

In [ ]:
import time
from statistics import median

TRIALS = 3

strategies = {
    "Standard (low effort)": (standard_client, standard_request),
    "MR1 (minimal effort)": (mr1_client, mr1_request),
}
for client, make_request in strategies.values():
    client.retrieve(make_request())

for label, (client, make_request) in strategies.items():
    timings = []
    for _ in range(TRIALS):
        start = time.perf_counter()
        client.retrieve(make_request())
        timings.append((time.perf_counter() - start) * 1000)
    print(f"{label}: median {median(timings):.0f} ms (trials: {', '.join(f'{t:.0f}' for t in timings)})")

## ✅ Mission complete

**What you built:**

* ✓ **Indexed knowledge sources**: One per document index, connected with the `chunk` field and PDF citation metadata.
* ✓ **Standard knowledge base**: Plans queries across all three sources and writes a grounded, cited answer.
* ✓ **MR1 knowledge base**: Searches all three sources without a chat completion model, for faster retrieval.

Your knowledge sources and knowledge bases stay in place for the next notebooks.

➡️ Return to lab instructions.